In [ ]:
import pandas as pd
from collections import Counter
import json

# 0. Load data mentah untuk dapatkan min/max tiap fitur
raw = pd.read_csv("transformation_segmentation_v4.csv")
# Kolom di raw: "Total Game", "Total Playtime", "Total Achievement"
min_game, max_game = raw["Total Game"].min(), raw["Total Game"].max()
min_play, max_play = raw["Total Playtime"].min(), raw["Total Playtime"].max()
min_ach,  max_ach  = raw["Total Achievement"].min(), raw["Total Achievement"].max()

# 1. Load data hasil segmentasi akhir (nilai sudah dinormalisasi 0–1)
df = pd.read_csv("archetypal_segments_elbowK_top3.csv")
# Pastikan kolom-nya bernama total_game, total_playtime, total_achievement
# dan berisi nilai 0–1

# 2. Tentukan jumlah arketipe dari kolom weight
K = sum(c.startswith("archetype_") and c.endswith("_weight") for c in df.columns)
weight_cols = [f"archetype_{k+1}_weight" for k in range(K)]

# 3. Pilih arketipe dominan per baris
df["dominant_archetype"] = df[weight_cols].idxmax(axis=1)

# 4. Inverse transform ke skala asli
def inv_scale(norm, min_, max_):
    return norm * (max_ - min_) + min_

df["total_game_orig"]       = df["total_game"].apply(lambda x: round(inv_scale(x, min_game, max_game), 2))
df["total_playtime_orig"]   = df["total_playtime"].apply(lambda x: round(inv_scale(x, min_play, max_play), 2))
df["total_achievement_orig"] = df["total_achievement"].apply(lambda x: round(inv_scale(x, min_ach, max_ach), 2))

# 5. Group berdasarkan arketipe dominan dan hitung rata-rata skala asli
results = []
json_output = {}

for ark, subset in df.groupby("dominant_archetype"):
    avg_game       = subset["total_game_orig"].mean().round(2)
    avg_playtime   = subset["total_playtime_orig"].mean().round(2)
    avg_achievement= subset["total_achievement_orig"].mean().round(2)

    # Mode topik (skip -1 dan 0 jika perlu)
    dom_topic = subset["dominant_topic"]
    dom_topic = dom_topic[~dom_topic.isin([-1, 0])] if dom_topic.notna().any() else dom_topic
    dominant_topic = dom_topic.mode().iloc[0] if not dom_topic.mode().empty else None

    # Hitung Top 3 genre dari kolom top_1_genre, top_2_genre, top_3_genre
    all_genres = pd.concat([
        subset["top_1_genre"],
        subset["top_2_genre"],
        subset["top_3_genre"]
    ])
    top3_genres = [str(g) for g, _ in Counter(all_genres.dropna()).most_common(3)]

    # Simpan ke list dan ke JSON
    results.append({
        "Archetype":             ark,
        "Avg_Game_Orig":         avg_game,
        "Avg_Playtime_Orig":     avg_playtime,
        "Avg_Achievement_Orig":  avg_achievement,
        "Dominant_Topic":        dominant_topic,
        "Top_3_Genre":           top3_genres
    })
    json_output[ark] = {
        "average_game_owned":    avg_game,
        "average_playtime":      avg_playtime,
        "average_achievement":   avg_achievement,
        "dominant_topic":        str(dominant_topic),
        "top_3_genres":          top3_genres
    }

# 6. Simpan ke CSV & JSON
df_result = pd.DataFrame(results)
df_result.to_csv("karakteristik_arketipe_scaled_back.csv", index=False)

with open("karakteristik_arketipe_llm_scaled_back.json", "w") as f:
    json.dump(json_output, f, indent=4)

print("✅ Inverse scaling dan penyimpanan selesai:")
print("   • karakteristik_arketipe_scaled_back.csv")
print("   • karakteristik_arketipe_llm_scaled_back.json")



✅ Inverse scaling dan penyimpanan selesai:
   • karakteristik_arketipe_scaled_back.csv
   • karakteristik_arketipe_llm_scaled_back.json


In [4]:
import pandas as pd
from collections import Counter
import json

# 1. Load karakteristik arketipe hasil inverse scaling
df = pd.read_csv("karakteristik_arketipe_scaled_back.csv")

# 2. Load mapping keywords per topic dari CSV
topic_kw_df = pd.read_csv("daftar_topik_keywords_automerged.csv")
# Bentuk dict: topic_id (int) -> list of keywords
topic_keywords = {
    int(row['topic']): row['keywords'].split(', ')
    for _, row in topic_kw_df.iterrows()
}

# 3. Load mapping genre code ke nama genre dari CSV
genre_map_df = pd.read_csv("genre_code_mapping.csv")
# Asumsi kolom: Genre, Genre_Code
genre_map = {
    str(row['Genre_Code']): row['Genre']
    for _, row in genre_map_df.iterrows()
}

# 4. Siapkan output JSON
json_output = {}
for _, row in df.iterrows():
    ark = row['Archetype']
    avg_game = row['Avg_Game_Orig']
    avg_playtime = row['Avg_Playtime_Orig']
    avg_achievement = row['Avg_Achievement_Orig']
    dom_topic_id = int(row['Dominant_Topic']) if pd.notna(row['Dominant_Topic']) else None

    # Ambil keywords topik dari mapping
    topic_entry = {
        "id": dom_topic_id,
        "keywords": topic_keywords.get(dom_topic_id, [])
    }

    # Map top_3_genres kode ke nama di dalam loop
    genre_codes = row['Top_3_Genre'].strip('[]').replace("'", "").split(',')
    genre_codes = [c.strip() for c in genre_codes if c.strip()]
    genres = [genre_map.get(c, c) for c in genre_codes]

    # Susun dict per arketipe
    json_output[ark] = {
        "average_game_owned": avg_game,
        "average_playtime": avg_playtime,
        "average_achievement": avg_achievement,
        "dominant_topic": topic_entry,
        "top_3_genres": genres
    }

# 5. Simpan JSON
target_file = "karakteristik_arketipe_enriched.json"
with open(target_file, "w", encoding="utf-8") as f:
    json.dump(json_output, f, indent=4, ensure_ascii=False)

print(f"✅ JSON with enriched topics and genre names saved to '{target_file}'")


✅ JSON with enriched topics and genre names saved to 'karakteristik_arketipe_enriched.json'


In [4]:
print(df.columns.tolist())

['steam_id', 'total_game', 'total_achievement', 'dominant_topic', 'top_1_genre', 'top_2_genre', 'top_3_genre', 'archetype_1_weight', 'archetype_2_weight', 'archetype_3_weight', 'archetype_4_weight', 'archetype_5_weight', 'archetype_6_weight', 'archetype_7_weight', 'dominant_archetype']
